# Cross-Species LPS Dataset Explorer

This notebook inspects the two h5ad files used in the CellOT cross-species experiment:
- `datasets/scrna-crossspecies/hvg-top1k-train-only.h5ad` — the training data (mouse subset, top-1000 HVGs)
- `datasets/valid_species.h5ad` — the validation / full-species dataset
- `results/cross_species/evals_iid_data_space/imputed.h5ad` — the model's transported (predicted) cells

In [1]:
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = (8, 4)

## 1. Training dataset: `hvg-top1k-train-only.h5ad`

In [3]:
adata = anndata.read_h5ad('datasets/scrna-crossspecies/hvg-top1k-train-only.h5ad')
print(adata)
print('\nShape:', adata.shape, '  (n_cells x n_genes)')

AnnData object with n_obs × n_vars = 62114 × 6619
    obs: 'condition', 'species', 'individual', 'batch', 'louvain', 'n_counts'
    var: 'gene_ids-0-0-0-0', 'gene_ids-1-0-0-0', 'gene_ids-2-0-0-0', 'gene_ids-0-1-0-0', 'gene_ids-1-1-0-0', 'gene_ids-2-1-0-0', 'gene_ids-0-0-1-0', 'gene_ids-1-0-1-0', 'gene_ids-2-0-1-0', 'gene_ids-0-1-1-0', 'gene_ids-1-1-1-0', 'gene_ids-2-1-1-0', 'gene_ids-3-1-1-0', 'gene_ids-0-0-0-1', 'gene_ids-1-0-0-1', 'gene_ids-2-0-0-1', 'gene_ids-0-1-0-1', 'gene_ids-1-1-0-1', 'gene_ids-2-1-0-1', 'gene_ids-0-0-1-1', 'gene_ids-1-0-1-1', 'gene_ids-2-0-1-1', 'gene_ids-0-1-1-1', 'gene_ids-1-1-1-1', 'gene_ids-2-1-1-1'
    uns: 'condition_colors', 'neighbors', 'species_colors'
    obsm: 'X_pca', 'X_umap'
    obsp: 'distances', 'connectivities'

Shape: (62114, 6619)   (n_cells x n_genes)


In [4]:
print('--- obs columns (cell metadata) ---')
print(adata.obs.columns.tolist())
print('\n--- var columns (gene metadata) ---')
print(adata.var.columns.tolist())
print('\n--- uns keys ---')
print(list(adata.uns.keys()))
print('\n--- obsm keys ---')
print(list(adata.obsm.keys()))

--- obs columns (cell metadata) ---
['condition', 'species', 'individual', 'batch', 'louvain', 'n_counts']

--- var columns (gene metadata) ---
['gene_ids-0-0-0-0', 'gene_ids-1-0-0-0', 'gene_ids-2-0-0-0', 'gene_ids-0-1-0-0', 'gene_ids-1-1-0-0', 'gene_ids-2-1-0-0', 'gene_ids-0-0-1-0', 'gene_ids-1-0-1-0', 'gene_ids-2-0-1-0', 'gene_ids-0-1-1-0', 'gene_ids-1-1-1-0', 'gene_ids-2-1-1-0', 'gene_ids-3-1-1-0', 'gene_ids-0-0-0-1', 'gene_ids-1-0-0-1', 'gene_ids-2-0-0-1', 'gene_ids-0-1-0-1', 'gene_ids-1-1-0-1', 'gene_ids-2-1-0-1', 'gene_ids-0-0-1-1', 'gene_ids-1-0-1-1', 'gene_ids-2-0-1-1', 'gene_ids-0-1-1-1', 'gene_ids-1-1-1-1', 'gene_ids-2-1-1-1']

--- uns keys ---
['condition_colors', 'neighbors', 'species_colors']

--- obsm keys ---
['X_pca', 'X_umap']


In [5]:
print('=== Species breakdown ===')
print(adata.obs['species'].value_counts())

print('\n=== Condition breakdown ===')
print(adata.obs['condition'].value_counts())

print('\n=== Species x Condition cross-tab ===')
print(pd.crosstab(adata.obs['species'], adata.obs['condition']))

=== Species breakdown ===
rat       17844
rabbit    15963
mouse     15053
pig       13254
Name: species, dtype: int64

=== Condition breakdown ===
unst    33350
LPS6    28764
Name: condition, dtype: int64

=== Species x Condition cross-tab ===
condition  LPS6  unst
species              
mouse      7428  7625
pig        4806  8448
rabbit     7107  8856
rat        9423  8421


In [6]:
# Show all unique conditions to understand LPS timepoints
print('All unique condition labels:')
print(sorted(adata.obs['condition'].unique()))

All unique condition labels:
['LPS6', 'unst']


In [7]:
# What does the obs dataframe look like?
adata.obs.head(10)

,condition,species,individual,batch,louvain,n_counts
index,,,,,,
AAACCTGTCGGTCTAA-1-0-0-0-0,LPS6,rabbit,3,0,7,1776.0
AAACCTGTCTGTCCGT-1-0-0-0-0,LPS6,rabbit,3,0,9,2722.0
AAACGGGAGCCGATTT-1-0-0-0-0,LPS6,rabbit,3,0,3,1396.0
AAACGGGAGGCGCTCT-1-0-0-0-0,LPS6,rabbit,3,0,3,1591.0
AAACGGGCACAACGCC-1-0-0-0-0,LPS6,rabbit,3,0,9,1341.0
AAACGGGGTTAAGTAG-1-0-0-0-0,LPS6,rabbit,3,0,3,1776.0
AAAGATGAGATCGGGT-1-0-0-0-0,LPS6,rabbit,3,0,3,2119.0
AAAGATGAGCTAACTC-1-0-0-0-0,LPS6,rabbit,3,0,3,1954.0
AAAGCAAAGGCTAGCA-1-0-0-0-0,LPS6,rabbit,3,0,3,1061.0


In [8]:
# First gene names
print('First 20 gene names:', adata.var_names[:20].tolist())
print('Total genes:', adata.n_vars)

First 20 gene names: ['Gnai3', 'Narf', 'Cav2', 'Klf6', 'Scmh1', 'Cox5a', 'Xpo6', 'Tfe3', 'Gna12', 'Pih1d2', 'Dlat', 'Sdhd', 'Ccnd2', 'Gpr107', 'Lhx2', 'Gmpr', 'Trim25', 'Scpep1', 'Hddc2', 'Pemt']
Total genes: 6619


In [9]:
# Is the data sparse or dense? What are value ranges?
import scipy.sparse as sparse
X = adata.X
if sparse.issparse(X):
    print('Matrix is SPARSE')
    print('Sparsity:', 1 - X.nnz / (X.shape[0]*X.shape[1]))
    X_dense = X.toarray()
else:
    print('Matrix is DENSE')
    X_dense = X

print('Value range: [{:.3f}, {:.3f}]'.format(X_dense.min(), X_dense.max()))
print('Mean:', X_dense.mean().round(4))
print('Likely log-normalized:', X_dense.max() < 20)

Matrix is SPARSE
Sparsity: 0.8129722324161497
Value range: [0.000, 7.383]
Mean: 0.179
Likely log-normalized: True


## 2. Mouse cells only: unst vs LPS6

In [10]:
# Subset to mouse cells
mouse = adata[adata.obs['species'] == 'mouse'].copy()
print('Mouse cells total:', len(mouse))
print(mouse.obs['condition'].value_counts())

Mouse cells total: 15053
unst    7625
LPS6    7428
Name: condition, dtype: int64


In [ ]:
# Check if LPS6 exists
has_lps6 = 'LPS6' in mouse.obs['condition'].values
has_unst = 'unst' in mouse.obs['condition'].values
print(f'LPS6 condition present: {has_lps6}')
print(f'unst condition present: {has_unst}')

if not has_lps6:
    print('\nWARNING: LPS6 not found. Available conditions:')
    print(sorted(mouse.obs['condition'].unique()))

LPS6 condition present: True
unst condition present: True


: 

In [ ]:
# Compare gene expression distributions: unst vs LPS6 in mouse
if has_lps6 and has_unst:
    unst_cells = mouse[mouse.obs['condition'] == 'unst']
    lps6_cells = mouse[mouse.obs['condition'] == 'LPS6']

    unst_mean = np.asarray(unst_cells.X.mean(axis=0)).flatten()
    lps6_mean = np.asarray(lps6_cells.X.mean(axis=0)).flatten()

    # Find top differentially expressed genes
    diff = lps6_mean - unst_mean
    top_up_idx = np.argsort(diff)[-15:][::-1]
    top_down_idx = np.argsort(diff)[:15]

    print('Top 15 upregulated genes (LPS6 vs unst):')
    for i in top_up_idx:
        print(f'  {adata.var_names[i]:12s}  Δmean={diff[i]:+.4f}')

    print('\nTop 15 downregulated genes:')
    for i in top_down_idx:
        print(f'  {adata.var_names[i]:12s}  Δmean={diff[i]:+.4f}')

In [ ]:
# Plot mean expression: unst vs LPS6 for top DE genes
if has_lps6 and has_unst:
    top_de_idx = np.argsort(np.abs(diff))[-20:][::-1]
    top_genes = adata.var_names[top_de_idx].tolist()

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(top_genes))
    ax.bar(x - 0.2, unst_mean[top_de_idx], width=0.4, label='unst', color='steelblue', alpha=0.8)
    ax.bar(x + 0.2, lps6_mean[top_de_idx], width=0.4, label='LPS6', color='firebrick', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(top_genes, rotation=45, ha='right')
    ax.set_ylabel('Mean expression')
    ax.set_title('Top 20 DE genes: unst vs LPS6 (mouse)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Full validation dataset: `valid_species.h5ad`

In [ ]:
valid = anndata.read_h5ad('datasets/valid_species.h5ad')
print(valid)
print('\n=== Species ===')
print(valid.obs['species'].value_counts())
print('\n=== Conditions ===')
print(valid.obs['condition'].value_counts())
print('\n=== Species x Condition ===')
print(pd.crosstab(valid.obs['species'], valid.obs['condition']))

## 4. Imputed (predicted) cells from CellOT

In [ ]:
import os
imputed_path = 'results/cross_species/evals_iid_data_space/imputed.h5ad'
if os.path.exists(imputed_path):
    imputed = anndata.read_h5ad(imputed_path)
    print('Imputed cells shape:', imputed.shape)
    print(imputed.obs.head())
else:
    print('imputed.h5ad not found at', imputed_path)

In [ ]:
# Compare imputed vs observed LPS6 cells
if os.path.exists(imputed_path) and has_lps6 and has_unst:
    lps6_obs_mean = lps6_mean  # from above: observed LPS6 mean per gene
    imp_arr = np.asarray(imputed.X.toarray() if sparse.issparse(imputed.X) else imputed.X)
    imp_mean = imp_arr.mean(axis=0)

    # r2 correlation between observed and imputed gene means
    corr = np.corrcoef(lps6_obs_mean, imp_mean)[0, 1]
    print(f'Pearson r (obs vs imputed gene means): {corr:.4f}')
    print(f'r² = {corr**2:.4f}')

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(lps6_obs_mean, imp_mean, alpha=0.3, s=5, color='teal')
    lims = [min(lps6_obs_mean.min(), imp_mean.min()),
            max(lps6_obs_mean.max(), imp_mean.max())]
    ax.plot(lims, lims, 'r--', lw=1, label='y=x')
    ax.set_xlabel('Observed LPS6 (mean per gene)')
    ax.set_ylabel('CellOT imputed (mean per gene)')
    ax.set_title(f'Gene mean: observed vs predicted\nr² = {corr**2:.3f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Quick sanity check: data split labels

Verifies that train/test/eval labels were applied correctly to the training dataset.

In [ ]:
# Check if split column was baked into the h5ad
if 'split' in adata.obs.columns:
    print('Split column found in obs:')
    print(adata.obs['split'].value_counts())
else:
    print('No split column in obs — split is assigned dynamically by the data loader')

if 'transport' in adata.obs.columns:
    print('\nTransport column (source/target assignment):')
    print(adata.obs['transport'].value_counts())